PHASE 1

In [82]:
import pandas as pd
import glob
import os
# combine all payment(BOLT) csv files into one file and creating a dataframe
files= glob.glob(r'C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\BOLT\bolt_payment\*.csv')
df_bolt_payments_raw = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
df_bolt_payments_raw.to_csv(r'C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\BOLT\bolt_payment\bolt_payment_stacked.csv', index=False)



In [83]:
# combine all ride(BOLT) csv files into one file and creating a dataframe 
path= r'C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\BOLT\Order history\order_history'
files = glob.glob(path + "/**/*.csv", recursive=True)
df_bolt_trips_raw = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
df_bolt_trips_raw.to_csv(r'C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\BOLT\Order history\order_history_stacked.csv', index=False)

In [84]:
# read the Uber trips csv file and creating a dataframe
path = r'C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\Uber\uber_trips.csv'
df_uber_trips_raw= pd.read_csv(path)

# read the Uber payments csv file and creating a dataframe
path = r"C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\Uber\uber_payments.csv"
df_uber_payments_raw= pd.read_csv(path)


In [85]:
import pyodbc
import pandas as pd

from sqlalchemy import create_engine

conn = pyodbc.connect(
    "Driver={ODBC Driver 18 for SQL Server};"
    "Server=LAMOSKI;"
    "Database=RideMetrics;"
    "Trusted_Connection=yes;"
    "Encrypt=no;"
)
engine = create_engine("mssql+pyodbc://LAMOSKI/RideMetrics?driver=ODBC+Driver+18+for+SQL+Server&trusted_connection=yes&encrypt=no")


cursor = conn.cursor()
cursor.execute("SELECT @@VERSION")
print(cursor.fetchone())

('Microsoft SQL Server 2022 (RTM) - 16.0.1000.6 (X64) \n\tOct  8 2022 05:58:25 \n\tCopyright (C) 2022 Microsoft Corporation\n\tDeveloper Edition (64-bit) on Windows 10 Home 10.0 <X64> (Build 26200: ) (Hypervisor)\n',)


REMOVAL OF PRIVATE SENSITIVE COLUMNS

In [86]:
# ingesting data into Bronze schema after prunning(private sensitive data) and selecting relevant columns.
#  Also saving the prunned dataframes as csv files in the RawFiles folder for future reference.

# Bolt payment data
df_bolt_payments_bronze = df_bolt_payments_raw[['Date', 'Payment method', 'Date of ride',  'Price (no VAT)', 'VAT', 'Price Total']]
df_bolt_payments_bronze.to_sql('Bolt_Payments', engine,schema='Bronze', if_exists= 'replace', index= False )
df_bolt_payments_bronze.to_csv(r'C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\RawFiles\Bolt_Payments.csv', index=False) 

# Bolt trips data
df_bolt_trips_bronze = df_bolt_trips_raw[['Time', 'Driver accepted', 'Ride finished', 'Pickup distance', 'Ride distance', 'Ride duration', 
                               'Order state', 'Reject reason', 'Pickup duration', 
                               'Estimated pickup time', 'Estimated distance', 'Estimated duration']]
df_bolt_trips_bronze.to_sql('Bolt_Trips', engine,schema= 'Bronze', if_exists= 'replace', index=False)
df_bolt_trips_bronze.to_csv(r'C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\RawFiles\Bolt_Trips.csv', index=False) 

# Uber trips data
df_uber_trips_bronze = df_uber_trips_raw[['global_product_name', 'status', 'request_timestamp_local',
                                'begintrip_timestamp_local', 'dropoff_timestamp_local',
                                'trip_distance_miles', 'trip_duration_seconds', 'base_fare_local',
                                'original_fare_local', 'cancellation_fee_local']]
df_uber_trips_bronze.to_sql('Uber_Trips', engine,schema='Bronze', if_exists= 'replace', index= False )
df_uber_trips_bronze.to_csv(r'C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\RawFiles\Uber_Trips.csv', index=False)

# Uber payments data
df_uber_payments_bronze = df_uber_payments_raw[['City Name', 'Trip UUID', 'Local Amount', 'Currency Code',
       'Classification', 'Category', 'Local Timestamp']]
df_uber_payments_bronze.to_sql('Uber_Payments', engine,schema='Bronze', if_exists= 'replace', index= False )
df_uber_payments_bronze.to_csv(r'C:\Users\olami\OneDrive\Desktop\LODZ RIDE METRICS\RawFiles\Uber_Payments.csv', index=False)


PHASE 2 (REMOVAL OF UNWANTED COLUMNS)

In [87]:
# reading the data from the Bronze schema to create dataframes for further processing in Silver and Gold layers.
df_bolt_trips_silver = pd.read_sql('select * from Bronze.Bolt_Trips',engine)
df_uber_trips_silver = pd.read_sql('select * from Bronze.Uber_Trips',engine)
df_uber_payments_silver = pd.read_sql('select * from Bronze.Uber_Payments',engine)

In [88]:
# selecting relevant columns for analysis
df_bolt_trips_silver = df_bolt_trips_silver[['Time', 'Driver accepted', 'Ride finished', 
       'Ride distance', 'Ride duration', 'Order state'
       ]]
df_uber_trips_silver = df_uber_trips_silver[[ 'status', 'request_timestamp_local',
       'begintrip_timestamp_local', 'dropoff_timestamp_local',
       'trip_distance_miles', 'trip_duration_seconds', 'base_fare_local',
       'original_fare_local', 'cancellation_fee_local']]
df_uber_payments_silver = df_uber_payments_silver[[ 'City Name', 'Trip UUID', 'Local Amount', 
       'Classification', 'Category', 'Local Timestamp']]



STANDADIZING DATETIME COLUMNS

In [89]:
# converting the Uber trips timestamp columns to datetime format for further analysis.
uber_trips_datetime_cols = ['request_timestamp_local', 'begintrip_timestamp_local',
       'dropoff_timestamp_local']
for i in uber_trips_datetime_cols:
       df_uber_trips_silver[i] = pd.to_datetime(df_uber_trips_silver[i], format='%Y-%m-%dT%H:%M:%S.%fZ',
        utc=True).dt.tz_localize(None)
       
#converting the Uber payments timestamp column to datetime format for further analysis.
df_uber_payments_silver['Local Timestamp'] = pd.to_datetime(df_uber_payments_silver['Local Timestamp'], format='mixed', errors='coerce')

# converting the Bolt trips timestamp columns to datetime format for further analysis.
bolt_trips_datetime_cols = ['Time', 'Driver accepted', 'Ride finished']
for i in bolt_trips_datetime_cols:
       df_bolt_trips_silver[i] = pd.to_datetime(df_bolt_trips_silver[i], format='%Y-%m-%d %H:%M:%S')




RENAMING COLUMNS FOR USER FRIENDLY

In [90]:
# Uber trips
df_uber_trips_silver.rename(columns={
    'status': 'trip_status',
    'request_timestamp_local': 'request_time',
    'begintrip_timestamp_local': 'start_time',
    'dropoff_timestamp_local': 'end_time',
    'trip_distance_miles': 'distance_miles',
    'trip_duration_seconds': 'duration_seconds',
    'base_fare_local': 'base_fare',
    'original_fare_local': 'gross_fare',
    'cancellation_fee_local': 'cancellation_fee'
}, inplace=True)

# Uber payments
df_uber_payments_silver.rename(columns={
    'City Name': 'city_name',
    'Trip UUID': 'trip_uuid',
    'Local Amount': 'amount',
    'Classification': 'classification',
    'Category': 'category',
    'Local Timestamp': 'local_timestamp'
}, inplace=True)

# Bolt trips
df_bolt_trips_silver.rename(columns={
    'Time': 'request_time',
    'Driver accepted': 'driver_accepted_time',
    'Ride finished': 'ride_finished_time',
    'Ride distance': 'ride_distance',
    'Ride duration': 'ride_duration',
    'Order state': 'order_state'
}, inplace=True)



DROPPING CANCELLED TRIPS


In [91]:
# Dropping cancelled trips from the Bolt trips dataframe to ensure that only completed trips are considered for analysis.
df_bolt_silver = df_bolt_trips_silver.dropna().copy()

PIVOTING UBER PAYMENT TABLE

In [92]:
# pivoting the Uber payments dataframe to have separate columns for each classification, 
# with the sum of amounts for each classification as the values.
df_uber_payments_silver_pivoted = df_uber_payments_silver.pivot_table(
    index=['local_timestamp', 'city_name'],
    columns='classification',
    values='amount',
    aggfunc='sum'
).fillna(0).reset_index()

# Fix column flattening
df_uber_payments_silver_pivoted.columns = [
    '_'.join(col).strip('_') if isinstance(col, tuple) else col
    for col in df_uber_payments_silver_pivoted.columns
]

# Define columns to exclude
exclude_cols = [
    'local_timestamp',
    'city_name',
    'transport.fare.cash.collected',
    'transport.misc.tip'
]

# Get all columns to sum
cols_to_sum = [col for col in df_uber_payments_silver_pivoted.columns 
               if col not in exclude_cols]

# Net earnings calculation: sum of all payment classifications except for cash collected.
df_uber_payments_silver_pivoted['net_earnings_no_tips'] = (
    df_uber_payments_silver_pivoted[cols_to_sum].sum(axis=1)
)
df_uber_payments_silver_pivoted['tips'] = (
    df_uber_payments_silver_pivoted['transport.misc.tip']
)

df_uber_payments_silver_pivoted['net_earnings_with_tips'] = df_uber_payments_silver_pivoted['net_earnings_no_tips']+ df_uber_payments_silver_pivoted['tips']

# keeping only the local_timestamp and net_earnings columns for further analysis. 
df_uber_payments_silver_pivoted = df_uber_payments_silver_pivoted[
    ['local_timestamp', 'net_earnings_no_tips', 'tips', 'net_earnings_with_tips']
]


In [93]:
df_uber_payments_silver_pivoted.head(3)

,local_timestamp,net_earnings_no_tips,tips,net_earnings_with_tips
0,2024-08-01 16:52:00,9.38,0.0,9.38
1,2024-08-01 17:01:00,13.43,0.0,13.43
2,2024-08-01 17:37:00,9.59,0.0,9.59


PRUNING MY DATAFRAME TO LODZ ONLY 


In [94]:
df_uber_payments_silver_pivoted = df_uber_payments_silver_pivoted[df_uber_payments_silver_pivoted['local_timestamp'] >= '2024-08-25']
df_uber_trips_silver= df_uber_trips_silver[df_uber_trips_silver['request_time'] >= '2024-08-25']

MERGE TABLES

In [95]:


# flooring the timestamp columns to the nearest minute for both Uber trips and payments dataframes to facilitate joining on a common time basis.
df_uber_trips_silver['request_time_rounded'] = df_uber_trips_silver['request_time'].dt.floor('min')
df_uber_payments_silver_pivoted['local_timestamp_rounded'] = df_uber_payments_silver_pivoted['local_timestamp'].dt.floor('min')

# joning the Uber trips and payments dataframes on the rounded timestamp columns to create a consolidated dataframe for analysis.
#automatically dropping the cancelled trips with no payments from the Uber trips dataframe during the join operation.
df_uber_silver = pd.merge(
    df_uber_trips_silver,
    df_uber_payments_silver_pivoted,
    left_on='request_time_rounded',
    right_on='local_timestamp_rounded',
    how='inner'
)





COMBINED TABLES(UBER AND BOLT)

In [96]:
df_uber_silver['platform'] = 'Uber'
df_bolt_silver['platform'] = 'Bolt' 

# converting uber distance duration column to metre to correspond with the Bolt distance duration column for further analysis.
df_uber_silver['distance_metres'] = (df_uber_silver['distance_miles'] * 1609.34).round(2)
df_uber_silver = df_uber_silver[['platform','request_time','end_time','distance_metres','duration_seconds',
                                'net_earnings_no_tips','tips','net_earnings_with_tips','gross_fare','trip_status']] 


In [100]:
uber_common = df_uber_silver[['platform',
    'request_time',
    'end_time',
    'distance_metres',
    'duration_seconds',
    'trip_status']].rename(columns={
        'distance_metres': 'distance',
        'duration_seconds': 'duration'
    })
bolt_common = df_bolt_silver[['platform',
    'request_time',
    'ride_finished_time',
    'ride_distance',
    'ride_duration',
    'order_state']].rename(columns={
        'ride_finished_time': 'end_time',
        'ride_distance': 'distance',
        'ride_duration': 'duration',
        'order_state': 'trip_status'
    })
df_combined_silver = pd.concat([uber_common, bolt_common], ignore_index=True)
df_combined_silver.to_sql('Combined_Silver', engine,schema='Silver', if_exists= 'replace', index= False )
df_uber_silver.to_sql('Uber', engine,schema='Silver', if_exists= 'replace', index= False )
df_bolt_silver.to_sql('Bolt', engine,schema='Silver', if_exists= 'replace', index= False )

101

In [101]:
df_uber_silver.head(3)

,platform,request_time,end_time,distance_metres,duration_seconds,net_earnings_no_tips,tips,net_earnings_with_tips,gross_fare,trip_status
0,Uber,2024-08-25 01:21:19,2024-08-25 01:43:25,6595.09,689.0,15.58,0.0,15.58,16.59,completed
1,Uber,2024-08-25 01:35:54,2024-08-25 02:09:40,7534.58,904.0,16.88,0.0,16.88,17.99,completed
2,Uber,2024-08-25 02:00:09,2024-08-25 02:22:26,4231.43,451.0,13.69,0.0,13.69,14.59,completed


In [99]:
df_combined_silver.head(3)

,platform,request_time,end_time,distance,duration,trip_status
0,Uber,2024-08-25 01:21:19,2024-08-25 01:43:25,6595.09,689.0,completed
1,Uber,2024-08-25 01:35:54,2024-08-25 02:09:40,7534.58,904.0,completed
2,Uber,2024-08-25 02:00:09,2024-08-25 02:22:26,4231.43,451.0,completed
